In [1]:
# ============================================================
# 08_calibration.ipynb
# Model Calibration Assessment
#
# Methods:
#   1. Calibration curve (reliability diagram)
#   2. Expected Calibration Error (ECE)
#   3. Hosmer-Lemeshow goodness-of-fit test
#   4. Brier Score decomposition
# ============================================================


# ─────────────────────────────────────────────
# Cell 1 | Imports
# ─────────────────────────────────────────────
import os
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss
from sklearn.model_selection import train_test_split
from scipy import stats

warnings.filterwarnings('ignore')

BASE_OUT = os.path.join('..', 'outputs')
BASE_MOD = os.path.join('..', 'outputs', 'models')
BASE_FIG = os.path.join('..', 'outputs', 'figures')

DISEASE_REGISTRY = {
    'htn': {
        'target'      : 'HE_HP',
        'label'       : 'Hypertension',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_NA', 'N_K'],
    },
    'dm': {
        'target'      : 'HE_DM_HbA1c',
        'label'       : 'Diabetes',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_SUGAR', 'N_CHO', 'N_EN'],
    },
    'dys': {
        'target'      : 'HE_HCHOL',
        'label'       : 'Dyslipidemia',
        'feature_cols': ['BD1_11', 'BD2_1', 'BS3_1',
                         'BE3_75', 'BE3_91', 'pa_aerobic',
                         'L_BR_FQ', 'BH1',
                         'HE_BMI', 'HE_wc', 'HE_wt',
                         'N_FAT', 'N_CHOL'],
    },
}

NON_ACTIONABLE = [
    'ID', 'year', 'age_group', 'age', 'sex',
    'edu', 'incm', 'ho_incm',
    'BE3_71', 'BE3_81', 'BP1', 'mh_stress',
    'HE_obe', 'BO1_1', 'BO1_2', 'BO1_3',
]
RANDOM_SEED = 42
N_BINS      = 10

print("Configuration loaded.")


# ─────────────────────────────────────────────
# Cell 2 | Calibration Metrics
# ─────────────────────────────────────────────
def expected_calibration_error(y_true, y_prob,
                                n_bins=10):
    """
    Expected Calibration Error (ECE).
    Weighted mean absolute difference between
    predicted probability and actual frequency.
    """
    bins    = np.linspace(0, 1, n_bins + 1)
    ece     = 0.0
    n_total = len(y_true)

    for i in range(n_bins):
        lo, hi = bins[i], bins[i+1]
        mask   = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n_total) * abs(acc - conf)

    return round(ece, 4)


def hosmer_lemeshow(y_true, y_prob, n_groups=10):
    """
    Hosmer-Lemeshow goodness-of-fit test.
    H0: model is well-calibrated.
    p > 0.05 indicates good calibration.
    """
    df_hl = pd.DataFrame(
        {'y': y_true, 'p': y_prob}
    ).sort_values('p')

    n      = len(df_hl)
    groups = np.array_split(df_hl, n_groups)

    hl_stat = 0.0
    for grp in groups:
        o1 = grp['y'].sum()
        e1 = grp['p'].sum()
        o0 = len(grp) - o1
        e0 = len(grp) - e1
        if e1 > 0:
            hl_stat += (o1 - e1)**2 / e1
        if e0 > 0:
            hl_stat += (o0 - e0)**2 / e0

    p_val = 1 - stats.chi2.cdf(
        hl_stat, df=n_groups - 2
    )
    return round(hl_stat, 3), round(p_val, 4)


def brier_decompose(y_true, y_prob):
    """
    Brier score decomposition:
      BS = Uncertainty - Resolution + Reliability
    """
    bs         = brier_score_loss(y_true, y_prob)
    prevalence = y_true.mean()
    uncertainty = prevalence * (1 - prevalence)
    return {
        'Brier_Score'  : round(bs, 4),
        'Uncertainty'  : round(uncertainty, 4),
        'Skill_Score'  : round(
            1 - bs / uncertainty, 4
        ) if uncertainty > 0 else np.nan,
    }


# ─────────────────────────────────────────────
# Cell 3 | Main Calibration Loop
# ─────────────────────────────────────────────
cal_rows = []

fig, axes = plt.subplots(
    1, 3, figsize=(15, 5), sharey=False
)

for ax, (d_key, d_info) in zip(
        axes, DISEASE_REGISTRY.items()):

    target = d_info['target']
    label  = d_info['label']
    feat   = d_info['feature_cols']

    df = pd.read_csv(
        os.path.join(BASE_OUT, f"{d_key}_total.csv")
    )
    drop  = NON_ACTIONABLE + [target]
    fcols = [c for c in df.columns
              if c not in drop and c in feat]
    X = df[fcols].astype(float)
    y = df[target].astype(int)

    _, X_test, _, y_test = train_test_split(
        X, y, test_size=0.2,
        stratify=y, random_state=RANDOM_SEED
    )

    model = joblib.load(
        os.path.join(BASE_MOD,
                     f"xgb_{d_key}_total.joblib")
    )
    y_prob = model.predict_proba(X_test)[:, 1]

    # Calibration curve
    prob_true, prob_pred = calibration_curve(
        y_test, y_prob, n_bins=N_BINS
    )

    # Metrics
    ece       = expected_calibration_error(
        y_test.values, y_prob, N_BINS
    )
    hl_stat, hl_p = hosmer_lemeshow(
        y_test.values, y_prob
    )
    brier_dec = brier_decompose(
        y_test.values, y_prob
    )

    cal_rows.append({
        'Disease'    : label,
        'ECE'        : ece,
        'HL_stat'    : hl_stat,
        'HL_p'       : hl_p,
        'HL_result'  : 'Good' if hl_p > 0.05
                       else 'Poor',
        **brier_dec,
    })

    print(f"\n  {label}:")
    print(f"    ECE         : {ece:.4f}")
    print(f"    HL test     : χ²={hl_stat:.3f}  "
          f"p={hl_p:.4f}  "
          f"({'Good' if hl_p > 0.05 else 'Poor'})")
    print(f"    Brier Score : "
          f"{brier_dec['Brier_Score']:.4f}")
    print(f"    Skill Score : "
          f"{brier_dec['Skill_Score']:.4f}")

    # Plot
    ax.plot(prob_pred, prob_true,
            marker='o', linewidth=2,
            color='#2c7bb6',
            label='XGBoost')
    ax.plot([0, 1], [0, 1], 'k--',
            linewidth=1, label='Perfect calibration')
    ax.fill_between(
        prob_pred,
        prob_true - 0.05,
        prob_true + 0.05,
        alpha=0.1, color='#2c7bb6'
    )
    ax.set_title(label, fontsize=13,
                 fontweight='bold')
    ax.set_xlabel('Mean Predicted Probability',
                  fontsize=10)
    ax.set_ylabel('Fraction of Positives',
                  fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.text(0.05, 0.85,
            f"ECE={ece:.3f}\n"
            f"HL p={hl_p:.3f}\n"
            f"Skill={brier_dec['Skill_Score']:.3f}",
            transform=ax.transAxes,
            fontsize=9,
            bbox=dict(boxstyle='round',
                      facecolor='wheat',
                      alpha=0.4))
    ax.legend(fontsize=8)
    ax.grid(linestyle='--', alpha=0.35)
    ax.spines[['top', 'right']].set_visible(False)

fig.suptitle(
    'Calibration Curves — XGBoost Underwriting Models\n'
    '(Reliability Diagrams, 10 bins, test set)',
    fontsize=12, y=1.03
)
plt.tight_layout()
fig_path = os.path.join(
    BASE_FIG, 'fig_calibration_curves.png'
)
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.close()
print(f"\nCalibration figure saved : {fig_path}")

df_cal = pd.DataFrame(cal_rows)
print("\n" + "="*60)
print("  CALIBRATION SUMMARY TABLE")
print("="*60)
print(df_cal.to_string(index=False))

cal_path = os.path.join(BASE_OUT,
                         'table_calibration.csv')
df_cal.to_csv(cal_path, index=False)
print(f"\nCalibration table saved : {cal_path}")
print("\n=== 08_calibration.ipynb complete ===")

Configuration loaded.

  Hypertension:
    ECE         : 0.1316
    HL test     : χ²=292.434  p=0.0000  (Poor)
    Brier Score : 0.1925
    Skill Score : 0.1182

  Diabetes:
    ECE         : 0.1592
    HL test     : χ²=492.297  p=0.0000  (Poor)
    Brier Score : 0.1616
    Skill Score : 0.1160

  Dyslipidemia:
    ECE         : 0.1654
    HL test     : χ²=558.009  p=0.0000  (Poor)
    Brier Score : 0.2163
    Skill Score : -0.0243

Calibration figure saved : ..\outputs\figures\fig_calibration_curves.png

  CALIBRATION SUMMARY TABLE
     Disease    ECE  HL_stat  HL_p HL_result  Brier_Score  Uncertainty  Skill_Score
Hypertension 0.1316  292.434   0.0      Poor       0.1925       0.2183       0.1182
    Diabetes 0.1592  492.297   0.0      Poor       0.1616       0.1828       0.1160
Dyslipidemia 0.1654  558.009   0.0      Poor       0.2163       0.2112      -0.0243

Calibration table saved : ..\outputs\table_calibration.csv

=== 08_calibration.ipynb complete ===
